# Pipeline RAG Chatbot Kampus UHO

Notebook ini membangun basis pengetahuan (knowledge base) untuk chatbot kampus dari dua sumber:
1. **Scraping** halaman web resmi (daftar URL di `URL/url-uho.txt`)
2. **PDF** peraturan/panduan akademik di folder `DATA_PDF/`, dengan sumber URL asal tiap PDF diambil dari `URL/url-sumber-pdf.txt`

Alur: scrape/ekstrak -> bersihkan teks -> chunking -> embedding -> simpan ke ChromaDB -> uji retrieval + generation dengan Groq (jawaban menyertakan daftar sumber).

**Struktur folder yang dipakai (di Google Drive):**
```
DATA UHO/
  URL/
    url-uho.txt          # daftar URL halaman untuk di-scrape
    url-sumber-pdf.txt   # mapping nama file PDF -> URL asal
  DATA_PDF/
    FMIPA/*.pdf          # PDF khusus fakultas, dikelompokkan per subfolder
    FT/*.pdf
    FHIL/*.pdf
    ...
    *.pdf                # PDF level universitas (langsung di root DATA_PDF/, tanpa subfolder) -> fakultas='UNIVERSITAS'
  chroma_db_akademik/          # dibuat otomatis, tempat vector store disimpan
```

## 1. Instalasi dependensi

In [20]:
import requests
!pip install -q -U \
    langchain \
    langchain-text-splitters \
    langchain-core \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    beautifulsoup4 \
    sentence-transformers \
    python-dotenv \
    tavily-python \
    pymupdf \
    requests==2.32.4 \
    opentelemetry-api==1.42.0 \
    opentelemetry-sdk==1.42.0 \
    opentelemetry-exporter-otlp-proto-grpc==1.42.0

## 2. Konfigurasi
Mount Google Drive, tentukan path folder, dan set API key Groq.

In [21]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os
from pathlib import Path

data_path = "/content/drive/MyDrive/DATA UHO"

url_folder_path = data_path + "/URL"
pdf_folder_path = data_path + "/DATA_PDF"          # nama folder PDF yang benar (bukan "/PDF")

url_uho_path = os.path.join(url_folder_path, "url-uho.txt")
pdf_source_map_path = os.path.join(url_folder_path, "url-sumber-pdf.txt")

PDF_DIR = Path(pdf_folder_path)
CHROMA_DIR = data_path + "/chroma_db_akademik"

print("File daftar URL scraping :", url_uho_path)
print("File mapping sumber PDF  :", pdf_source_map_path)
print("Folder PDF               :", PDF_DIR.resolve())
print("Vector store             :", CHROMA_DIR)

File daftar URL scraping : /content/drive/MyDrive/DATA UHO/URL/url-uho.txt
File mapping sumber PDF  : /content/drive/MyDrive/DATA UHO/URL/url-sumber-pdf.txt
Folder PDF               : /content/drive/MyDrive/DATA UHO/DATA_PDF
Vector store             : /content/drive/MyDrive/DATA UHO/chroma_db_akademik


In [23]:
# API key Groq -- diambil dari Colab Secrets (ikon kunci di sidebar kiri), bukan hardcode di notebook.
# Buat secret baru bernama GROQ_API_KEY berisi key kamu, lalu aktifkan akses untuk notebook ini.
# Buat juga secret TAVILY_API_KEY (daftar gratis di tavily.com) untuk fallback web search live.
#
# PENTING: kalau kamu sempat menempel API key asli langsung di notebook/chat sebelumnya,
# revoke key lama itu di console.groq.com dan buat key baru sebelum melanjutkan.
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("GROQ_API_KEY berhasil dimuat." if os.environ.get("GROQ_API_KEY") else "GROQ_API_KEY belum diset!")
print("TAVILY_API_KEY berhasil dimuat." if os.environ.get("TAVILY_API_KEY") else "TAVILY_API_KEY belum diset!")

GROQ_API_KEY berhasil dimuat.
TAVILY_API_KEY berhasil dimuat.


## 3. Scraping halaman web (dari `url-uho.txt`)

In [24]:
import re

url_pattern = re.compile(r'https?://[^\s"\',\]]+')

with open(url_uho_path, "r", encoding="utf-8") as f:
    content = f.read()

URLS = list(dict.fromkeys(url_pattern.findall(content)))  # ekstrak URL + buang duplikat, urutan tetap
print(f"Total URL ditemukan di url-uho.txt: {len(URLS)}")
URLS[:5]

Total URL ditemukan di url-uho.txt: 256


['https://uho.ac.id/',
 'https://satudata.uho.ac.id/',
 'https://accounting.uho.ac.id',
 'https://admin.e-election.uho.ac.id',
 'https://admin.kkn.uho.ac.id']

In [25]:
import requests
from bs4 import BeautifulSoup
import time

def scrape_page(url: str, timeout: int = 10) -> dict:
    """Ambil teks utama dari satu halaman web."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; UHO-RAG-Bot/1.0)"}
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()

    title = soup.title.string.strip() if soup.title and soup.title.string else url
    text = soup.get_text(separator="\n")

    return {"source": url, "title": title, "raw_text": text}


scraped_docs = []
for url in URLS:
    try:
        doc = scrape_page(url)
        scraped_docs.append(doc)
        print(f"[OK] {url} -> {len(doc['raw_text'])} karakter")
    except Exception as e:
        print(f"[GAGAL] {url}: {e}")
    time.sleep(1)  # jeda sopan antar request

print(f"\nTotal halaman berhasil di-scrape: {len(scraped_docs)}")

[OK] https://uho.ac.id/ -> 9055 karakter
[OK] https://satudata.uho.ac.id/ -> 79 karakter
[GAGAL] https://accounting.uho.ac.id: 403 Client Error: Forbidden for url: https://accounting.uho.ac.id/
[GAGAL] https://admin.e-election.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.e-election.uho.ac.id/
[GAGAL] https://admin.kkn.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.kkn.uho.ac.id/
[GAGAL] https://admin.kkntematik.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.kkntematik.uho.ac.id/
[GAGAL] https://admin.ofc.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.ofc.uho.ac.id/
[GAGAL] https://admin.regsm-tahapke2.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.regsm-tahapke2.uho.ac.id/
[GAGAL] https://admin.remun.uho.ac.id: 403 Client Error: Forbidden for url: https://admin.remun.uho.ac.id/
[GAGAL] https://admin.siakadbeta.uho.ac.id: HTTPSConnectionPool(host='admin.siakadbeta.uho.ac.id', port=443): Max retries exceeded wit

## 4. Ekstraksi teks dari PDF (dengan sumber URL dari `url-sumber-pdf.txt`)

In [26]:
def parse_pdf_source_map(path_txt: str) -> dict:
    """Baca url-sumber-pdf.txt (blok dipisah baris kosong, format 'nama file: ...' dan 'url : ...')
    dan kembalikan dict {nama_file_pdf: url_asal}."""
    with open(path_txt, "r", encoding="utf-8") as f:
        content = f.read()

    mapping = {}
    blocks = re.split(r"\n\s*\n", content.strip())
    for block in blocks:
        nama_match = re.search(r"nama file\s*:\s*(.+)", block, re.IGNORECASE)
        url_match = re.search(r"url\s*:\s*(.+)", block, re.IGNORECASE)
        if nama_match and url_match:
            mapping[nama_match.group(1).strip()] = url_match.group(1).strip()

    return mapping


pdf_source_map = parse_pdf_source_map(pdf_source_map_path)
print(f"Total mapping PDF -> URL ditemukan: {len(pdf_source_map)}")
pdf_source_map

Total mapping PDF -> URL ditemukan: 3


{'PERATURAN REKTOR UHO ttg Pengelolaan SDM.pdf': 'https://uho.ac.id/wp-content/uploads/2023/10/Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO.pdf',
 'PERREK_TARIF_2025.pdf': 'https://uho.ac.id/peraturan-rektor-universitas-halu-oleo-nomor-20-tahun-2025-tentang-tarif-layanan-badan-layanan-umum-universitas-halu-oleo/',
 'Peraturan-Rektor-UHO-Tentang-Perlindungan-Data-Pribadi-di-Lingkungan-Universitas-Halu-Oleo.pdf': 'https://uho.ac.id/peraturan-rektor-universitas-halu-oleo-tentang-perlindungan-data-pribadi-di-lingkungan-universitas-halu-oleo/'}

In [27]:
import fitz  # PyMuPDF

def extract_pdf(path: Path, source_map: dict, root_dir: Path) -> dict:
    """Ekstrak seluruh teks dari satu file PDF. Sumber diambil dari mapping URL kalau nama file cocok,
    kalau tidak ada di mapping fallback ke path lokal file. Fakultas diambil dari nama subfolder
    langsung di bawah DATA_PDF/ (mis. DATA_PDF/FMIPA/x.pdf -> fakultas='FMIPA'); kalau PDF ada
    langsung di root DATA_PDF/ tanpa subfolder, fakultas diberi label 'UNIVERSITAS'."""
    text_parts = []
    with fitz.open(path) as pdf:
        for page in pdf:
            text_parts.append(page.get_text())
    url_asal = source_map.get(path.name, str(path))

    relative = path.relative_to(root_dir)
    fakultas = relative.parts[0] if len(relative.parts) > 1 else "UNIVERSITAS"

    return {
        "source": url_asal,
        "title": path.stem,
        "raw_text": "\n".join(text_parts),
        "fakultas": fakultas,
    }


pdf_docs = []
pdf_files = list(PDF_DIR.rglob("*.pdf"))  # rglob supaya ikut menyusuri subfolder per fakultas
for pdf_path in pdf_files:
    try:
        doc = extract_pdf(pdf_path, pdf_source_map, PDF_DIR)
        pdf_docs.append(doc)
        print(f"[OK] [{doc['fakultas']}] {pdf_path.name} -> {len(doc['raw_text'])} karakter (sumber: {doc['source']})")
    except Exception as e:
        print(f"[GAGAL] {pdf_path.name}: {e}")

print(f"\nTotal PDF berhasil diekstrak: {len(pdf_docs)}")
if not pdf_files:
    print("Tidak ada file PDF ditemukan di", PDF_DIR.resolve(), "- cek kembali nama foldernya.")

pdf_tanpa_mapping = [p.name for p in pdf_files if p.name not in pdf_source_map]
if pdf_tanpa_mapping:
    print("\n[PERINGATAN] PDF berikut tidak ada di url-sumber-pdf.txt, sumbernya fallback ke path lokal:")
    for nama in pdf_tanpa_mapping:
        print(" -", nama)

print("\nRingkasan jumlah PDF per fakultas:")
from collections import Counter
for fakultas, jumlah in Counter(d["fakultas"] for d in pdf_docs).items():
    print(f" - {fakultas}: {jumlah}")

[OK] [UNIVERSITAS] UHO Siapkan Peraturan Akademik yang Lebih Adaptif - UHO.pdf -> 2467 karakter (sumber: /content/drive/MyDrive/DATA UHO/DATA_PDF/UHO Siapkan Peraturan Akademik yang Lebih Adaptif - UHO.pdf)
[OK] [UNIVERSITAS] Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO.pdf -> 133172 karakter (sumber: /content/drive/MyDrive/DATA UHO/DATA_PDF/Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO.pdf)
[OK] [UNIVERSITAS] PERREK_TARIF_2025.pdf -> 59 karakter (sumber: https://uho.ac.id/peraturan-rektor-universitas-halu-oleo-nomor-20-tahun-2025-tentang-tarif-layanan-badan-layanan-umum-universitas-halu-oleo/)
[OK] [UNIVERSITAS] Peraturan-Rektor-UHO-Tentang-Perlindungan-Data-Pribadi-di-Lingkungan-Universitas-Halu-Oleo.pdf -> 12 karakter (sumber: https://uho.ac.id/peraturan-rektor-universitas-halu-oleo-tentang-perlindungan-data-pribadi-di-lingkungan-universitas-halu-oleo/)
[OK] [UNIVERSITAS] kebijakan-rektor-universitas-halu-oleo-terkait-peraturan-akademik-di-

## 5. Pembersihan teks

In [28]:
def clean_text(text: str) -> str:
    """Bersihkan whitespace berlebih dan baris kosong berulang."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]
    return "\n".join(lines)


all_docs = scraped_docs + pdf_docs
for doc in all_docs:
    doc["clean_text"] = clean_text(doc["raw_text"])

print(f"Total dokumen (web + PDF) setelah pembersihan: {len(all_docs)}")

Total dokumen (web + PDF) setelah pembersihan: 80


## 6. Chunking dengan metadata sumber

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import datetime

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\nBAB ", "\nPasal ", "\n\n", "\n", ". ", " ", ""],  # prioritaskan potongan per Pasal/BAB untuk dokumen legal
)

chunked_documents = []
today = datetime.date.today().isoformat()

for doc in all_docs:
    chunks = splitter.split_text(doc["clean_text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "source": doc["source"],
                    "title": doc["title"],
                    "chunk_index": i,
                    "ingested_at": today,
                    "fakultas": doc.get("fakultas", "UNIVERSITAS"),  # scraping web = UNIVERSITAS, PDF = nama subfolder
                },
            )
        )

print(f"Total chunk siap di-embed: {len(chunked_documents)}")
if chunked_documents:
    print("\nContoh chunk pertama:\n", chunked_documents[0].page_content[:300])
    print("\nMetadata:", chunked_documents[0].metadata)

# Daftar judul dokumen unik -- dipakai nanti untuk deteksi "dokumen yang disebut di pertanyaan"
document_titles = list({doc["title"] for doc in all_docs})

Total chunk siap di-embed: 2004

Contoh chunk pertama:
 Beranda - UHO
Tentang
Berita
Satu Data UHO
Direktori MoU
Pendidikan
Penelitian & Pengabdian
Unit/Organisasi
Fakultas
Fakultas Keguruan dan Ilmu Pendidikan
Fakultas Ilmu sosial dan Ilmu Politik
Fakultas Matematika dan Ilmu Pengetahuan
Fakultas Perikanan dan Ilmu Kelautan
Fakultas Kehutanan dan Ilmu L

Metadata: {'source': 'https://uho.ac.id/', 'title': 'Beranda - UHO', 'chunk_index': 0, 'ingested_at': '2026-09-04', 'fakultas': 'UNIVERSITAS'}


## 7. Embedding dan penyimpanan ke ChromaDB
Menggunakan model embedding multilingual (mendukung bahasa Indonesia) dari sentence-transformers.

In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# vectorstore = Chroma.from_documents(
#     documents=chunked_documents,
#     embedding=embedding_model,
#     persist_directory=CHROMA_DIR,
#     collection_name="uho_knowledge_base",
# )
vectorstore = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name="uho_knowledge_base",
    collection_metadata={"hnsw:space": "cosine"},  # supaya relevance score konsisten di skala 0-1
)
# Catatan: versi Chroma terbaru auto-persist ke persist_directory, tidak perlu panggil .persist() lagi.

print(f"Vector store tersimpan di '{CHROMA_DIR}' dengan {len(chunked_documents)} chunk.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store tersimpan di '/content/drive/MyDrive/DATA UHO/chroma_db_akademik' dengan 2004 chunk.


## 8. Uji coba retrieval

In [31]:
query_uji = "berapa kali maksimal mahasiswa boleh mengajukan cuti akademik"
hasil = vectorstore.similarity_search(query_uji, k=3)

for i, r in enumerate(hasil, 1):
    print(f"--- Hasil {i} (sumber: {r.metadata.get('title')} | {r.metadata.get('source')}) ---")
    print(r.page_content[:300])
    print()

--- Hasil 1 (sumber: kebijakan-rektor-universitas-halu-oleo-terkait-peraturan-akademik-di-universitas-halu-oleo-1720147116-unNjzUNXt6 | /content/drive/MyDrive/DATA UHO/DATA_PDF/kebijakan-rektor-universitas-halu-oleo-terkait-peraturan-akademik-di-universitas-halu-oleo-1720147116-unNjzUNXt6.pdf) ---
karena tindakan/hukuman/sanksi akademik, maka waktu
penghentian sementara tersebut diperhitungkan dalam batas
lama studi maksimum.
(7)
Untuk dapat diterima kembali sebagai mahasiswa aktif setelah
menjalani masa cuti akademik yaitu dengan cara yang
bersangkutan kembali membayar SPP/UKT semester yang


--- Hasil 2 (sumber: Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO | /content/drive/MyDrive/DATA UHO/DATA_PDF/Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO.pdf) ---
Dekan/Direktur.
(5)
Mahasiswa yang disetujui menjalani cuti akademik, tidak
diperhitungkan dalam batas waktu studi maksimum yang masih
tersedia baginya dan tidak dikenakan pembayaran SPP/UKT.


## 8b. Diagnosis cepat: cek apakah teks tertentu benar-benar terekstrak dari PDF

In [32]:
kata_kunci_cek = "Pasal 1"
for doc in pdf_docs:
    ada = kata_kunci_cek in doc["clean_text"]
    print(f"[{'OK' if ada else 'TIDAK ADA'}] '{kata_kunci_cek}' di dokumen: {doc['title']}")

[TIDAK ADA] 'Pasal 1' di dokumen: UHO Siapkan Peraturan Akademik yang Lebih Adaptif - UHO
[OK] 'Pasal 1' di dokumen: Peraturan-Rektor-Nomor-1-Tahun-2019-Tentang-Peraturan-Akademik-UHO
[TIDAK ADA] 'Pasal 1' di dokumen: PERREK_TARIF_2025
[TIDAK ADA] 'Pasal 1' di dokumen: Peraturan-Rektor-UHO-Tentang-Perlindungan-Data-Pribadi-di-Lingkungan-Universitas-Halu-Oleo
[OK] 'Pasal 1' di dokumen: kebijakan-rektor-universitas-halu-oleo-terkait-peraturan-akademik-di-universitas-halu-oleo-1720147116-unNjzUNXt6
[TIDAK ADA] 'Pasal 1' di dokumen: SK-Kalender-Akademik-2024-2025
[OK] 'Pasal 1' di dokumen: 3-PERATURAN-REKTOR-UHO-NOMOR-1-TAHUN-2021-TENTANG-KEBIJAKAN-MBKM (1)
[TIDAK ADA] 'Pasal 1' di dokumen: Pengembangan Kurikulum_Peraturan Rektor No 1 tahun 2022
[TIDAK ADA] 'Pasal 1' di dokumen: 1-SK Rektor_KALENDER-AKADEMIK-TAHUN-AKADEMIK-2023-2024
[TIDAK ADA] 'Pasal 1' di dokumen: 7-SK Penetapan Panitia Persiapan Akreditasi Internasional Prodi Di Lingkup UHO Tahun 2023
[OK] 'Pasal 1' di dokumen: 48 - Per

## 9. RAG generation hybrid: Vector store (utama) + Web search live (fallback)

Urutan prioritas: cari dulu di vector store (hasil scraping + PDF resmi yang sudah dikurasi). Web search live (dibatasi ke `site:uho.ac.id`) HANYA dipakai kalau skor kecocokan di vector store rendah -- supaya tidak menarik sumber pihak ketiga yang belum terverifikasi saat dokumen resmi sudah cukup menjawab.

In [33]:
from langchain_groq import ChatGroq
from tavily import TavilyClient
from urllib.parse import urlparse

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

SYSTEM_PROMPT = (
    "Kamu adalah asisten informasi kampus Universitas Halu Oleo (UHO). "
    "Jawab HANYA berdasarkan konteks yang diberikan. "
    "Jika konteks tidak cukup untuk menjawab, katakan dengan jujur bahwa informasi tidak tersedia "
    "dan sarankan mahasiswa menghubungi bagian akademik terkait. Jangan mengarang informasi. "
    "Kalau sebagian konteks berasal dari hasil pencarian web (ditandai '[Web]'), sebutkan di jawaban "
    "bahwa bagian itu adalah info tambahan dari pencarian web, bukan dari dokumen resmi yang sudah diverifikasi."
)

AMBANG_SKOR_RELEVANSI = 0.5  # sesuaikan berdasarkan pengalaman: makin rendah = makin jarang fallback ke web


def cari_dokumen_disebut(pertanyaan: str, titles: list, threshold: float = 0.3):
    """Deteksi apakah pertanyaan menyebut nama dokumen tertentu (mis. judul peraturan),
    supaya pencarian bisa difokuskan ke dokumen itu saja -- mencegah 'nyasar' ke dokumen
    lain yang topiknya mirip."""
    pertanyaan_lower = pertanyaan.lower()
    best_title, best_score = None, 0
    for title in titles:
        kata_judul = set(title.lower().replace("-", " ").replace("_", " ").split())
        overlap = sum(1 for w in kata_judul if len(w) > 3 and w in pertanyaan_lower)
        score = overlap / max(len(kata_judul), 1)
        if score > best_score:
            best_score, best_title = score, title
    return best_title if best_score >= threshold else None

def dapatkan_domain_terverifikasi() -> list:
    raw = vectorstore.get(include=["metadatas"])
    domains = {urlparse(m["source"]).netloc for m in raw.get("metadatas", []) if m.get("source", "").startswith("http")}
    return list(domains)

DOMAIN_TERVERIFIKASI = dapatkan_domain_terverifikasi()
print(f"Total domain terverifikasi: {len(DOMAIN_TERVERIFIKASI)}")
print(DOMAIN_TERVERIFIKASI)


def cari_web_uho(pertanyaan: str, max_results: int = 3) -> list:
    """Fallback web search live, dibatasi HANYA ke domain yang sudah terbukti berhasil
    di-scrape sebelumnya -- bukan menebak seluruh uho.ac.id lewat trik 'site:'."""
    try:
        hasil = tavily.search(
            query=pertanyaan,
            include_domains=DOMAIN_TERVERIFIKASI or ["uho.ac.id"],  # fallback kalau daftar kosong
            max_results=max_results,
        )
    except Exception as e:
        print(f"[Web search gagal] {e}")
        return []
    return [
        {"title": r.get("title", r["url"]), "source": r["url"], "content": r.get("content", "")}
        for r in hasil.get("results", [])
    ]


def jawab_pertanyaan(pertanyaan: str, k: int = 8) -> str:
    dokumen_disebut = cari_dokumen_disebut(pertanyaan, document_titles)
    filter_meta = {"title": dokumen_disebut} if dokumen_disebut else None

    hasil_dengan_skor = vectorstore.similarity_search_with_relevance_scores(
        pertanyaan, k=k, filter=filter_meta
    )
    hasil_lokal = [doc for doc, _ in hasil_dengan_skor]
    skor_tertinggi = max((s for _, s in hasil_dengan_skor), default=0)

    potongan_konteks = [
        f"[Sumber: {r.metadata.get('title')}]\n{r.page_content}" for r in hasil_lokal
    ]
    sumber_terpakai = [{"title": r.metadata.get("title"), "source": r.metadata.get("source"), "asal": "vector_store"} for r in hasil_lokal]

    web_dipakai = False
    if skor_tertinggi < AMBANG_SKOR_RELEVANSI:
        hasil_web = cari_web_uho(pertanyaan)
        if hasil_web:
            web_dipakai = True
            for w in hasil_web:
                potongan_konteks.append(f"[Web] [Sumber: {w['title']}]\n{w['content']}")
                sumber_terpakai.append({"title": w["title"], "source": w["source"], "asal": "web_search"})

    konteks = "\n\n".join(potongan_konteks)
    prompt = (
        f"Konteks:\n{konteks}\n\n"
        f"Pertanyaan mahasiswa: {pertanyaan}\n\n"
        "Jawab dengan jelas dan ringkas dalam Bahasa Indonesia."
    )
    respons = llm.invoke([
        ("system", SYSTEM_PROMPT),
        ("human", prompt),
    ])

    # Kumpulkan sumber unik (title + source + asal) tanpa duplikat
    sumber_unik, seen = [], set()
    for m in sumber_terpakai:
        key = m["source"]
        if key not in seen:
            seen.add(key)
            sumber_unik.append(m)

    daftar_sumber = "\n".join(
        f"- {m['title']} ({m['source']})" + ("  [dari web search]" if m["asal"] == "web_search" else "")
        for m in sumber_unik
    )

    catatan_web = "\n\n(Catatan: sebagian jawaban di atas memakai hasil pencarian web karena dokumen resmi lokal belum cukup relevan.)" if web_dipakai else ""

    return f"{respons.content}{catatan_web}\n\nSumber:\n{daftar_sumber}"


# print(jawab_pertanyaan("apa saja syarat pengajuan cuti akademik di UHO?"))
print(jawab_pertanyaan("siapa rektor pengganti sekarang?"))

Total domain terverifikasi: 19
['amalilmiah.uho.ac.id', 'spi.simanja.uho.ac.id', 'neorespublica.uho.ac.id', 'jiikpp.uho.ac.id', 'bkdr.uho.ac.id', 'prodi.bkdr.uho.ac.id', 'bastra.uho.ac.id', 'uho.ac.id', 'pasca.uho.ac.id', 'piston-jt.uho.ac.id', 'satudata.uho.ac.id', 'infobeasiswa.uho.ac.id', 'simanja.uho.ac.id', 'ekinerja.uho.ac.id', 'siakadbeta.uho.ac.id', 'admin.simanja.uho.ac.id', 'sipas.fmipa.uho.ac.id', 'fmipa.uho.ac.id', 'holrev.uho.ac.id']


/tmp/ipykernel_3831/1381757867.py:66: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4490201f-e935-4b8a-acfe-1083eadd6897', metadata={'source': '/content/drive/MyDrive/DATA UHO/DATA_PDF/48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO;.pdf', 'chunk_index': 102, 'title': '48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO;', 'ingested_at': '2026-09-03'}, page_content='yang sah.\n(3) Penggantian\nanggota\nMahkamah\nSengketa\nOrganisasi\nKemahasiswaan yang berhenti sebagaimana dimaksud pada ayat\n(1) digantikan oleh calon anggota Mahkamah Sengketa Organisasi\nKemahasiswaan berdasarkan Keputusan Rektor.\nBAB IV\nPERSIDANGAN\nBagian Kesatu\nTata CaraPersidangan\nPasal 89\n(1) Persidangandilaksanakandalam ruangan di lingkungan Universitas\nHalu Oleo dengantertib aman,lancar, danberwibawa.\n(2) S ebelummemasukiruangsidang,para\npihak,saksi,danahliyanghadirwajib\nmengisidaftarhadir'), -8.3067747022

Rektor pengganti (Plt. Rektor) saat ini adalah **Prof. Dr. Khairul Munandi, S.T., M.Eng.**  
*(Informasi ini berasal dari hasil pencarian web, bukan dari dokumen resmi yang sudah diverifikasi.)*

(Catatan: sebagian jawaban di atas memakai hasil pencarian web karena dokumen resmi lokal belum cukup relevan.)

Sumber:
- 48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO; (/content/drive/MyDrive/DATA UHO/DATA_PDF/48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO;.pdf)
- Peraturan-Rektor-Kemahasiswaan (/content/drive/MyDrive/DATA UHO/DATA_PDF/FMIPA/Peraturan-Rektor-Kemahasiswaan.pdf)
- 48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO; (/content/drive/MyDrive/DATA UHO/DATA_PDF/UNIVERSITAS/48 - Peraturan Rektor UHO Nomor 853a Tahun 2015 tentang Organisasi Kemahasiswaan UHO;.pdf)
- REKTOR APRESIASI PELUNCURAN PEMBUKAAN PPDS ILMU BEDAH FK UHO - UHO (https://uho.ac.id/rektor-uho-hadiri-pelunc

## 10. Catatan lanjutan

- **Re-scrape berkala**: jadwalkan ulang cell scraping karena info seperti pengumuman dan kalender akademik sering berubah.
- **Update incremental**: kalau tidak ingin membangun ulang seluruh vector store, simpan hash/tanggal per dokumen dan hanya re-embed dokumen yang berubah.
- **Tambah PDF baru**: taruh file PDF di `DATA_PDF/`, lalu tambahkan bloknya (`nama file:` dan `url:`) di `url-sumber-pdf.txt` supaya sumbernya ikut tercatat sebagai URL, bukan path lokal.
- **Kalau jawaban 'tidak tersedia' padahal datanya ada**: naikkan `k` di `jawab_pertanyaan`, pastikan judul dokumen di `document_titles` cukup unik untuk terdeteksi `cari_dokumen_disebut`, atau jalankan sel diagnosis di bagian 8b untuk cek apakah teksnya benar-benar terekstrak dari PDF (kalau tidak, PDF-nya kemungkinan hasil scan dan butuh OCR, bukan ekstraksi teks biasa).
- **Web search fallback**: hanya aktif kalau skor relevansi vector store di bawah `AMBANG_SKOR_RELEVANSI` (default 0.5). Sesuaikan ambang ini setelah dicoba beberapa kali -- terlalu rendah berarti jarang fallback meski dokumen resmi kosong, terlalu tinggi berarti sering fallback padahal dokumen resmi sudah cukup. Web search dibatasi ke `site:uho.ac.id` supaya tidak menarik sumber pihak ketiga yang tidak terverifikasi.
- **Kalau 403/gagal terus saat scraping** halaman tertentu di bagian 3: coba header User-Agent browser asli dulu, atau tambahkan fallback Playwright (`page.goto` + `page.inner_text('body')`) untuk halaman yang butuh render JavaScript / menolak request `requests` biasa.
- **Dokumen mirip/duplikat** (mis. dua PDF tentang topik yang sama dari sumber berbeda) sebaiknya dicek manual dan disisakan satu saja, supaya tidak saling bersaing di hasil pencarian.
- **Filter per fakultas**: metadata `fakultas` sudah tersedia di tiap chunk (diambil dari nama subfolder di `DATA_PDF/`). Bisa dipakai untuk filter tambahan di `jawab_pertanyaan`, mis. `filter_meta = {"fakultas": "FMIPA"}`, kalau nanti perlu membatasi jawaban ke fakultas tertentu.
- **Kategori/collection terpisah**: kalau volume dokumen makin besar, pertimbangkan collection ChromaDB terpisah per topik (akademik, kemahasiswaan, umum) supaya retrieval lebih presisi.
- **Integrasi ke backend Flask Tanya KI**: fungsi `vectorstore` dan `jawab_pertanyaan` di notebook ini bisa dipindah langsung jadi modul Python yang dipanggil dari endpoint Flask yang sudah ada.